# 🚴 Green & Fast 2026 — Eco-Delivery
## Pipeline Data Science : Analyse & Prédiction du Temps de Livraison

**Client :** Naima, CEO Eco-Delivery  
**Equipe :** Nexa Data Science — M1 Data & IA  
**Dataset :** [Food Delivery Dataset — Kaggle](https://www.kaggle.com/datasets/gauravmalik26/food-delivery-dataset)

---

### Objectifs
1. **Analyser** les données pour comprendre les facteurs qui allongent les livraisons
2. **Identifier** les heures et zones à risque (trafic, délais)
3. **Construire** un modèle baseline pour prédire le temps de livraison
4. **Formuler** des recommandations opérationnelles pour Eco-Delivery

---

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Style global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

# Couleurs Eco-Delivery
GREEN  = '#2E7D32'
ORANGE = '#F57C00'
BLUE   = '#1565C0'
RED    = '#C62828'

print('Imports OK')

---
## 1. Chargement & Nettoyage des Données

In [ ]:
df_raw = pd.read_csv('train.csv')
df_raw.columns = df_raw.columns.str.strip()
print(f'Shape brute : {df_raw.shape}')
df_raw.head(3)

In [ ]:
df = df_raw.copy()

# Suppression des espaces parasites
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Variable cible : "(min) 24" → 24
df['Time_taken(min)'] = (
    df['Time_taken(min)']
    .str.replace(r'\(min\)\s*', '', regex=True)
    .str.strip()
    .astype(float)
)

# Météo : "conditions Sunny" → "Sunny"
df['Weatherconditions'] = df['Weatherconditions'].str.replace(r'^conditions\s*', '', regex=True)

# Conversions numériques
df['Delivery_person_Age']     = pd.to_numeric(df['Delivery_person_Age'],     errors='coerce')
df['Delivery_person_Ratings'] = pd.to_numeric(df['Delivery_person_Ratings'], errors='coerce')
df['multiple_deliveries']     = pd.to_numeric(df['multiple_deliveries'],     errors='coerce')

# Remplacement des NaN textuels 'NaN' par np.nan
df.replace('NaN', np.nan, inplace=True)

# Suppression des lignes sans cible
df = df.dropna(subset=['Time_taken(min)'])

# Imputation : médiane pour numérique, mode pour catégoriel
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f'Shape nettoyée : {df.shape}')
print(f'Valeurs manquantes résiduelles : {df.isna().sum().sum()}')

In [ ]:
# Visualisation des valeurs manquantes AVANT nettoyage
df_miss = df_raw.copy()
for col in df_miss.select_dtypes(include='object').columns:
    df_miss[col] = df_miss[col].str.strip()
df_miss.replace('NaN', np.nan, inplace=True)

missing_pct = (df_miss.isna().sum() / len(df_miss) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(missing_pct.index, missing_pct.values, color=ORANGE)
ax.bar_label(bars, fmt='%.1f%%', padding=4)
ax.set_xlabel('% valeurs manquantes')
ax.set_title('Valeurs manquantes par colonne (données brutes)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Analyse Descriptive Générale

In [ ]:
df.describe(include='all').T

In [ ]:
# Distribution des variables catégorielles clés
cat_cols = ['City', 'Road_traffic_density', 'Weatherconditions', 'Type_of_vehicle', 'Type_of_order']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color=sns.color_palette('muted', len(counts)))
    axes[i].set_title(col, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=30)
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 100, str(v), ha='center', fontsize=9)

axes[5].axis('off')
fig.suptitle('Distribution des variables catégorielles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Étude de la Variable Cible : `Time_taken(min)`

In [ ]:
target = df['Time_taken(min)']

print('=== Statistiques de la variable cible ===')
print(f"  Minimum  : {target.min():.0f} min")
print(f"  Maximum  : {target.max():.0f} min")
print(f"  Moyenne  : {target.mean():.1f} min")
print(f"  Médiane  : {target.median():.0f} min")
print(f"  Écart-type : {target.std():.1f} min")
print(f"  Skewness : {target.skew():.3f}  (légèrement asymétrique à droite)")
print(f"\n  % livraisons > 30 min (hors promesse) : {(target > 30).mean()*100:.1f}%")
print(f"  % livraisons <= 30 min (dans les clous) : {(target <= 30).mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme + KDE
axes[0].hist(target, bins=30, color=GREEN, edgecolor='white', alpha=0.85, density=True)
target.plot.kde(ax=axes[0], color=ORANGE, linewidth=2.5)
axes[0].axvline(target.mean(),   color=RED,  linestyle='--', linewidth=1.8, label=f'Moyenne : {target.mean():.1f} min')
axes[0].axvline(target.median(), color=BLUE, linestyle=':',  linewidth=1.8, label=f'Médiane : {target.median():.0f} min')
axes[0].axvline(30, color='black', linestyle='-', linewidth=1.5, alpha=0.6, label='Seuil 30 min (SLA)')
axes[0].set_title('Distribution du temps de livraison', fontweight='bold')
axes[0].set_xlabel('Temps (min)')
axes[0].legend()

# Boxplot par ville (trié par médiane)
order_city = df.groupby('City')['Time_taken(min)'].median().sort_values().index.tolist()
data_by_city = [df[df['City'] == c]['Time_taken(min)'].values for c in order_city]
axes[1].boxplot(data_by_city, labels=order_city,
                boxprops=dict(color=GREEN), medianprops=dict(color=RED, linewidth=2),
                whiskerprops=dict(color=GREEN), capprops=dict(color=GREEN))
axes[1].set_title('Temps de livraison par ville', fontweight='bold')
axes[1].set_xlabel('Ville')
axes[1].set_ylabel('Temps (min)')
plt.suptitle('')
plt.tight_layout()
plt.show()

print("\n>>> Semi-Urban affiche des temps nettement plus élevés (~50 min).")
print("    Cela s'explique par des distances plus longues et une densité de livreurs plus faible.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Par trafic
traffic_order = ['Low', 'Medium', 'High', 'Jam']
means_traffic = df[df['Road_traffic_density'].isin(traffic_order)].groupby('Road_traffic_density')['Time_taken(min)'].mean().reindex(traffic_order)
colors_t = [GREEN, BLUE, ORANGE, RED]
bars = axes[0].bar(means_traffic.index, means_traffic.values, color=colors_t, edgecolor='white')
axes[0].bar_label(bars, fmt='%.1f min', padding=3)
axes[0].axhline(30, color='black', linestyle='--', linewidth=1.2, alpha=0.5, label='SLA 30 min')
axes[0].set_title('Temps moyen par densité trafic', fontweight='bold')
axes[0].set_ylabel('Temps (min)')
axes[0].legend()

# Par météo
means_weather = df.groupby('Weatherconditions')['Time_taken(min)'].mean().sort_values(ascending=False)
bars2 = axes[1].bar(means_weather.index, means_weather.values, color=BLUE, alpha=0.8, edgecolor='white')
axes[1].bar_label(bars2, fmt='%.1f', padding=3)
axes[1].axhline(30, color='black', linestyle='--', linewidth=1.2, alpha=0.5)
axes[1].set_title('Temps moyen par météo', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

# Par véhicule
means_vehicle = df.groupby('Type_of_vehicle')['Time_taken(min)'].mean().sort_values(ascending=False)
bars3 = axes[2].bar(means_vehicle.index, means_vehicle.values, color=GREEN, alpha=0.8, edgecolor='white')
axes[2].bar_label(bars3, fmt='%.1f', padding=3)
axes[2].set_title('Temps moyen par type de véhicule', fontweight='bold')

fig.suptitle('Impact des facteurs opérationnels sur le temps de livraison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Analyse des Pics de Trafic & Moments Critiques

In [ ]:
# Extraction de l'heure de commande
df['order_hour'] = df['Time_Orderd'].str.split(':').str[0].astype(float)

# Temps moyen par heure
hourly_time = df.groupby('order_hour')['Time_taken(min)'].mean().sort_index()

# Volume de commandes par heure
hourly_vol = df.groupby('order_hour').size().sort_index()

# % de trafic dense (Jam + High) par heure
traffic_heavy = df[df['Road_traffic_density'].isin(['Jam', 'High'])]
traffic_pct = (traffic_heavy.groupby('order_hour').size() / hourly_vol * 100).fillna(0).sort_index()

print('Top 5 heures les plus chargées (temps moyen) :')
print(hourly_time.sort_values(ascending=False).head(5).to_string())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

hours = hourly_time.index

# ── Graphe 1 : Temps moyen par heure ──
colors_h = [RED if h in [19, 20, 21] else (ORANGE if h in [11, 12, 13, 14] else GREEN) for h in hours]
bars1 = axes[0].bar(hours, hourly_time.values, color=colors_h, edgecolor='white', width=0.8)
axes[0].axhline(30, color='black', linestyle='--', linewidth=1.5, alpha=0.6, label='SLA 30 min')
axes[0].axhline(hourly_time.mean(), color=BLUE, linestyle=':', linewidth=1.5, label=f'Moyenne : {hourly_time.mean():.1f} min')
axes[0].set_title('Temps de livraison moyen par heure de commande', fontweight='bold')
axes[0].set_ylabel('Temps (min)')
axes[0].set_xticks(range(0, 24))
axes[0].legend()
# Annotations pics
for h in [19, 20, 21]:
    if h in hourly_time.index:
        axes[0].annotate(f'⚠ {hourly_time[h]:.0f}min',
                         xy=(h, hourly_time[h]), xytext=(h, hourly_time[h]+1.5),
                         ha='center', color=RED, fontsize=8.5, fontweight='bold')

# ── Graphe 2 : Volume de commandes par heure ──
axes[1].bar(hours, hourly_vol.values, color=BLUE, alpha=0.75, edgecolor='white', width=0.8)
axes[1].set_title('Volume de commandes par heure', fontweight='bold')
axes[1].set_ylabel('Nombre de commandes')
axes[1].set_xticks(range(0, 24))

# ── Graphe 3 : % trafic Jam+High par heure ──
colors_t = [RED if v > 80 else (ORANGE if v > 20 else GREEN) for v in traffic_pct.values]
axes[2].bar(traffic_pct.index, traffic_pct.values, color=colors_t, edgecolor='white', width=0.8)
axes[2].axhline(80, color=RED, linestyle='--', linewidth=1.5, alpha=0.7, label='Seuil critique 80%')
axes[2].set_title('% de trafic Jam + High par heure', fontweight='bold')
axes[2].set_ylabel('% trafic dense')
axes[2].set_xlabel('Heure de la journée')
axes[2].set_xticks(range(0, 24))
axes[2].legend()

fig.suptitle('Analyse temporelle : Quand livre-t-on ? Quand est-ce difficile ?',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n>>> PICS CRITIQUES identifiés : 11h-14h (déjeuner) et 19h-21h (dîner)")
print("    Trafic 100% dense (Jam+High) entre 11h-14h et 19h-21h.")

In [ ]:
# Heatmap : temps moyen selon heure x ville
pivot = df.pivot_table(
    values='Time_taken(min)',
    index='City',
    columns='order_hour',
    aggfunc='mean'
).round(1)

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(
    pivot, annot=True, fmt='.0f', cmap='RdYlGn_r',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Temps moyen (min)'}
)
ax.set_title('Heatmap : Temps de livraison moyen (min) — Ville × Heure', fontweight='bold')
ax.set_xlabel('Heure de commande')
ax.set_ylabel('Ville')
plt.tight_layout()
plt.show()

print("\n>>> Semi-Urban est systématiquement en rouge : nécessite une stratégie dédiée.")

In [ ]:
# Distribution du trafic par ville
traffic_city = (
    df[df['Road_traffic_density'].isin(['Low','Medium','High','Jam'])]
    .groupby(['City','Road_traffic_density'])
    .size()
    .unstack(fill_value=0)
)
# Normaliser en %
traffic_city_pct = traffic_city.div(traffic_city.sum(axis=1), axis=0) * 100

traffic_city_pct[['Low','Medium','High','Jam']].plot(
    kind='bar', stacked=True, figsize=(10, 5),
    color=[GREEN, BLUE, ORANGE, RED], edgecolor='white'
)
plt.title('Répartition du trafic par ville (%)', fontweight='bold')
plt.ylabel('% des commandes')
plt.xticks(rotation=0)
plt.legend(title='Trafic', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

---
## 5. Matrice de Corrélation

In [ ]:
# Préparation : calcul distance et encodages pour la corrélation
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

df['distance_km'] = df.apply(
    lambda r: haversine_km(r['Restaurant_latitude'], r['Restaurant_longitude'],
                           r['Delivery_location_latitude'], r['Delivery_location_longitude']),
    axis=1
)

# Temps de préparation
def prep_time(t1, t2):
    try:
        p1, p2 = str(t1).split(':'), str(t2).split(':')
        h1, m1 = int(p1[0]), int(p1[1])
        h2, m2 = int(p2[0]), int(p2[1])
        diff = (h2*60+m2) - (h1*60+m1)
        return diff if diff >= 0 else diff + 1440
    except:
        return np.nan

df['prep_time_min'] = df.apply(lambda r: prep_time(r['Time_Orderd'], r['Time_Order_picked']), axis=1)
df['prep_time_min'] = df['prep_time_min'].fillna(df['prep_time_min'].median())

# Encodages
le = LabelEncoder()
for col in ['Weatherconditions','Road_traffic_density','Type_of_order','Type_of_vehicle','Festival','City']:
    df[col+'_enc'] = le.fit_transform(df[col].astype(str))

print('Features engineerées OK')

In [ ]:
corr_cols = [
    'Time_taken(min)', 'distance_km', 'prep_time_min', 'order_hour',
    'Delivery_person_Age', 'Delivery_person_Ratings',
    'Vehicle_condition', 'multiple_deliveries',
    'Road_traffic_density_enc', 'Weatherconditions_enc',
    'City_enc', 'Festival_enc', 'Type_of_vehicle_enc'
]

corr_matrix = df[corr_cols].corr()

# Renommage pour lisibilité
rename_map = {
    'Time_taken(min)':           'Temps livraison',
    'distance_km':               'Distance (km)',
    'prep_time_min':             'Temps prépa',
    'order_hour':                'Heure commande',
    'Delivery_person_Age':       'Age livreur',
    'Delivery_person_Ratings':   'Note livreur',
    'Vehicle_condition':         'Etat véhicule',
    'multiple_deliveries':       'Livraisons multiples',
    'Road_traffic_density_enc':  'Trafic',
    'Weatherconditions_enc':     'Météo',
    'City_enc':                  'Ville',
    'Festival_enc':              'Festival',
    'Type_of_vehicle_enc':       'Type véhicule'
}
corr_matrix.rename(index=rename_map, columns=rename_map, inplace=True)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # triangle supérieur masqué
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Matrice de Corrélation — Variables clés', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Corrélations avec la variable cible triées
corr_target = corr_matrix['Temps livraison'].drop('Temps livraison').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = [RED if v > 0 else BLUE for v in corr_target.values]
bars = ax.barh(corr_target.index, corr_target.values, color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.bar_label(bars, fmt='%.2f', padding=4, fontsize=9)
ax.set_title('Corrélation de chaque variable avec le Temps de Livraison', fontweight='bold')
ax.set_xlabel('Coefficient de corrélation de Pearson')
plt.tight_layout()
plt.show()

print("\n>>> Les variables les plus corrélées avec le temps de livraison :")
for name, val in corr_target.head(5).items():
    direction = 'augmente' if val > 0 else 'réduit'
    print(f"   • {name:25s} : {val:+.3f}  ({direction} le temps)")

---
## 6. Recommandations Opérationnelles pour Eco-Delivery

> Basées sur l'analyse des données 2024-2025

### 🔴 Recommandation 1 — Renforcer la flotte aux heures de pointe

**Constat :** Les créneaux **11h–14h** et **19h–21h** concentrent les pics de trafic (100% Jam/High) **et** les temps de livraison les plus longs (jusqu'à **31 min** en moyenne pour les commandes de 20h).

**Action :** Pré-positionner 30 à 40% de coursiers supplémentaires dès **10h30** et **18h30** selon le volume historique par quartier. Utiliser le modèle Random Forest pour prédire la demande H+1.

**Impact CO₂ :** Réduire les trajets à vide = réduire l'empreinte carbone par livraison.

---

### 🔴 Recommandation 2 — Stratégie dédiée Semi-Urban

**Constat :** Les villes **Semi-Urban** affichent un temps moyen de **~50 min**, soit presque le double des zones Urban/Metropolitan (~23–27 min). Ces zones ne peuvent pas tenir la promesse "30 min ou offert".

**Action :** Deux options :
- Exclure les zones Semi-Urban de la promesse 30 min (adapter le SLA à 45 min)
- Créer des **hubs de pré-stockage** (dark kitchens) dans ces zones pour réduire la distance

---

### 🟠 Recommandation 3 — Favoriser scooters électriques et vélos en heures de pointe

**Constat :** Les **scooters électriques** et **vélos** ont des temps moyens de **24–26 min**, contre **27–28 min** pour les motos en zone encombrée. En trafic dense, leur agilité compense la vitesse de pointe.

**Action :** Lors des pics de trafic urbain (Jam/High), router les nouvelles commandes vers les coursiers à vélo/scooter électrique disponibles en priorité.

**Impact CO₂ :** Direct — remplacement de motos thermiques par des véhicules zéro émission.

---

### 🟠 Recommandation 4 — Surveiller la météo Cloudy & Fog

**Constat :** Les conditions **Nuageux** et **Brouillard** allongent le temps moyen à **~29 min** (vs 22 min par temps ensoleillé), soit +7 minutes, risquant de dépasser le SLA.

**Action :** Intégrer un flux météo en temps réel dans le système de dispatch pour déclencher automatiquement des alertes et pré-appeler des renforts lors de prévisions Cloudy/Fog.

---

### 🟢 Recommandation 5 — Valoriser les livreurs bien notés

**Constat :** La note du livreur (`Delivery_person_Ratings`) est la **feature la plus importante** dans le Random Forest (20.8%). Les livreurs avec une note >= 4.8 livrent significativement plus vite.

**Action :** Mettre en place un système de gamification (bonus, priorité de dispatch) pour les livreurs bien notés, et un accompagnement pour ceux sous 4.2.

---

### 🟢 Recommandation 6 — Réduire les livraisons multiples aux heures de pic

**Constat :** Les livraisons multiples allongent le temps de livraison de manière significative, surtout aux heures de pic où le trafic est déjà dense.

**Action :** Désactiver les livraisons multiples (groupage) entre **11h–14h** et **19h–21h** en zones Urban/Metropolitan pour garantir le SLA.

---
## 7. Pipeline de Données & Feature Engineering

In [ ]:
FEATURES = [
    'Delivery_person_Age',
    'Delivery_person_Ratings',
    'Vehicle_condition',
    'multiple_deliveries',
    'distance_km',
    'prep_time_min',
    'order_hour',
    'Weatherconditions_enc',
    'Road_traffic_density_enc',
    'Type_of_order_enc',
    'Type_of_vehicle_enc',
    'Festival_enc',
    'City_enc',
]
TARGET = 'Time_taken(min)'

# Vérification
missing = [f for f in FEATURES if f not in df.columns]
print(f'Features disponibles : {len(FEATURES) - len(missing)}/{len(FEATURES)}')
if missing:
    print(f'Manquantes : {missing}')

---
## 8. Entraînement des Modèles Baseline

In [ ]:
X = df[FEATURES].copy()
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES)
y = df[TARGET].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train : {len(X_train)} lignes | Test : {len(X_test)} lignes')

In [ ]:
# ── Modèle 1 : Régression Linéaire (baseline simple) ──
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f'Régression Linéaire — MAE : {mae_lr:.2f} min | RMSE : {rmse_lr:.2f} min')

In [ ]:
# ── Modèle 2 : Random Forest ──
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print(f'Random Forest       — MAE : {mae_rf:.2f} min | RMSE : {rmse_rf:.2f} min')

---
## 9. Résultats & Visualisation des Performances

In [ ]:
results = pd.DataFrame({
    'Modèle'         : ['Régression Linéaire', 'Random Forest'],
    'MAE (min)'      : [round(mae_lr, 2),  round(mae_rf, 2)],
    'RMSE (min)'     : [round(rmse_lr, 2), round(rmse_rf, 2)],
    'MAE / SLA (%)'  : [round(mae_lr/30*100, 1), round(mae_rf/30*100, 1)]
})
results.set_index('Modèle', inplace=True)
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, name, color in [
    (axes[0], y_pred_lr, 'Régression Linéaire', BLUE),
    (axes[1], y_pred_rf, 'Random Forest',       GREEN)
]:
    ax.scatter(y_test, y_pred, alpha=0.15, s=8, color=color)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Prédiction parfaite')
    mae = mean_absolute_error(y_test, y_pred)
    ax.set_title(f'{name}\nMAE = {mae:.2f} min', fontweight='bold')
    ax.set_xlabel('Temps réel (min)')
    ax.set_ylabel('Temps prédit (min)')
    ax.legend(fontsize=9)

fig.suptitle('Valeurs réelles vs. prédites', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances Random Forest
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_fi = [RED if importances[f] == importances.max() else GREEN for f in importances.index]
bars = ax.barh(importances.index, importances.values * 100, color=colors_fi, edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=4)
ax.set_title('Importance des features — Random Forest', fontweight='bold')
ax.set_xlabel('Importance (%)')
plt.tight_layout()
plt.show()

print("\n>>> TOP 3 facteurs prédictifs :")
for feat, val in importances.sort_values(ascending=False).head(3).items():
    print(f"   {feat:30s} : {val*100:.1f}%")

---
## 10. Conclusion Sprint 1

| Critère                  | Résultat                          | Statut |
|--------------------------|-----------------------------------|--------|
| Pipeline de données      | 45 593 lignes nettoyées           | ✅     |
| Feature engineering      | 13 features dont distance GPS     | ✅     |
| Modèle baseline          | Random Forest — MAE 3.20 min      | ✅     |
| MAE < 8 min (DoD)        | 3.20 min ✓                        | ✅     |
| Recommandations métier   | 6 recommandations formulées       | ✅     |

### Prochaines étapes (Sprint 2)
- Hyperparameter tuning (GridSearchCV)
- Clustering géographique des zones de forte demande
- Dashboard interactif (Streamlit ou Plotly Dash)
- Intégration flux météo en temps réel